# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.


In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )

MODEL = "gpt-5-nano"
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

# why JSON? because its a structure that LLMs are familiar with. LLMs training data are usually - markdown (shorthand of HTML), natural language, JSON
# giving one example of a possible answer is known as one-shot prompting. multiple questions and example answers => multi-shot prompting
# this is one-shot prompting

In [5]:
# keep experimenting and iterating the prompts, that is how the 'Do not' section came about.
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
        # we can tell the AI to generate the format of the output generated.
        # this also constrains and forces it to generate json at INFERENCE time, only choose tokens that can form a VALID JSON. (without this, because the model is generating tokens based on a probability distribution, there is a POSSIBILITY that the model can generate tokens that will NOT form a valid json)
    )
    result = response.choices[0].message.content
    print(result)
    links = json.loads(result)  # turning json into python dictionary
    print(links)
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

{
  "links": [
    {"type": "home page", "url": "https://edwarddonner.com/"},
    {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
    {"type": "portfolio page", "url": "https://edwarddonner.com/curriculum/"},
    {"type": "skills page", "url": "https://edwarddonner.com/proficient/"},
    {"type": "project page", "url": "https://edwarddonner.com/connect-four/"},
    {"type": "project page", "url": "https://edwarddonner.com/outsmart/"},
    {"type": "blog page", "url": "https://edwarddonner.com/posts/"},
    {"type": "linkedin", "url": "https://www.linkedin.com/in/eddonner/"},
    {"type": "twitter", "url": "https://twitter.com/edwarddonner"},
    {"type": "facebook", "url": "https://www.facebook.com/edward.donner.52"}
  ]
}
{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'}, {'type': 'about page', 'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}, {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'}

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano


In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(
        url
    )  # this is just the AI call to get it to select relevant links (like agentic workflows - wrap LLM calls in a function and stiching them together.)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.6-35B-A3B
Updated
5 days ago
•
335k
•
1k
tencent/HY-Embodied-0.5
Updated
6 days ago
•
1.66k
•
881
unsloth/Qwen3.6-35B-A3B-GGUF
Updated
about 1 hour ago
•
816k
•
535
baidu/ERNIE-Image
Updated
3 days ago
•
4.14k
•
485
tencent/HY-World-2.0
Updated
4 days ago
•
485
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
623
OmniVoice
🌍
623
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.24k
Wan2.2 14B Preview
🐌
2.24k
generate a video from an image with a text prompt
Running
137
Bonsai 1-bit WebGPU
🌳


In [23]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# the 'without code blocks' is to ensure the AI does NOT generate markdown wrapped in ``

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[
        :5_000
    ]  # Truncate if more than 5,000 characters (can use 5000, 5_000 is for readability)
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.6-35B-A3B\nUpdated\n5 days ago\n•\n335k\n•\n1.01k\ntencent/HY-Embodied-0.5\nUpdated\n6 days ago\n•\n1.66k\n•\n881\nunsloth/Qwen3.6-35B-A3B-GGUF\nUpdated\nabout 1 hour ago\n•\n816k\n•\n535\nbaidu/ERNIE-Image\nUpdated\n3 days ago\n•\n4.14k\n•\n485\ntencent/HY-World-2.0\nUpdated\n4 days ago\n•\n485\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n623\nOmniVoice\n🌍

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        # we don't want json back, so the call is just model and messages
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

### The AI Community Building the Future

Hugging Face is a leading platform and community hub where machine learning engineers, data scientists, and AI enthusiasts collaborate to build the future of artificial intelligence. It is the home of open-source AI models, datasets, and innovative applications, empowering a global community to accelerate machine learning research and development.

---

### What Hugging Face Offers

- **2 Million+ Models & 500k+ Datasets:** Explore an extensive and ever-growing repository of machine learning models and datasets spanning text, image, video, audio, and even 3D modalities.
  
- **Spaces:** Run and showcase AI-powered applications in a community-driven hosting environment that supports state-of-the-art technologies like voice cloning, video generation, and local model execution on browsers or GPUs.
  
- **Collaboration Platform:** Host and collaborate on unlimited public projects. Share work, experiment, build portfolios, and stay at the cutting edge with the Hugging Face open-source stack.
  
- **Enterprise and Team Solutions:** Designed for organizations to securely scale AI projects, Hugging Face offers advanced enterprise-grade features including Single Sign-On (SSO), granular access controls, audit logs, increased compute options, private storage, and dedicated support.

---

### The Hugging Face Community & Culture

At its core, Hugging Face fosters an inclusive, collaborative, and ethical AI community that values openness, transparency, and innovation. The platform empowers the next generation of ML scientists, engineers, and end-users to learn, work together, and share knowledge. Hugging Face is proud of its fast-growing global community and talented science team who continuously explore and push the boundaries of AI technologies.

---

### Customers & Use Cases

Hugging Face supports a diverse range of users including individual researchers, academic institutions, startup teams, and global enterprises. Industries from healthcare to finance leverage its models and tools for natural language processing, computer vision, speech synthesis, and multimodal AI applications. Their enterprise customers benefit from scalable infrastructure, compliance with security standards, and analytics for managing AI deployments.

---

### Careers at Hugging Face

Join a pioneering AI company at the heart of the AI revolution. Hugging Face offers opportunities for passionate individuals eager to contribute to open-source innovation, build powerful ML tools, and make a real impact on the future of technology. The company values creativity, collaboration, and a commitment to building ethical AI.

- Work alongside top AI researchers and engineers
- Contribute to a growing open-source ecosystem
- Be part of a mission-driven team focused on community and technology

Explore current job openings and get involved in shaping AI’s next chapter.

---

### Brand & Visual Identity

Hugging Face's vibrant brand colors include:

- Yellow: #FFD21E
- Orange: #FF9D00
- Gray: #6B7280

Their iconic logos and brand assets reflect an energetic and welcoming community focused on innovation and openness.

---

### Connect & Learn More

- **Website:** https://huggingface.co  
- **GitHub:** [github.com/huggingface](https://github.com/huggingface)  
- **Twitter:** [@huggingface](https://twitter.com/huggingface)  
- **LinkedIn:** [Hugging Face](https://www.linkedin.com/company/hugging-face)  
- **Discord:** Join the community chats and forums for real-time collaboration and support.

---

### Summary

Hugging Face is transforming AI through openness, collaboration, and inclusive innovation. Whether you’re looking to explore millions of ML models, collaborate on cutting-edge datasets, build AI apps, or deploy at enterprise scale with security and control—Hugging Face offers a comprehensive, community-powered platform to accelerate and democratize artificial intelligence.

---

**Join Hugging Face today and help build the future of AI!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation


In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        stream=True,  # change stream to True, and you get back something called a stream, which you can iterate over as the data comes back from the api (its an iterator)
    )
    response = ""
    display_handle = display(
        Markdown(""), display_id=True
    )  # display an empty markdown, so that we can update the markdown as the token gets printed from the stream
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and collaboration platform dedicated to building the future of machine learning together. With a mission to empower the global machine learning community, Hugging Face creates a central hub where engineers, scientists, and end users can share, explore, and develop open-source models, datasets, and applications.

---

## What We Offer

- **Models:** Access over 2 million open-source ML models across various modalities including text, image, video, audio, and even 3D. Hugging Face hosts popular models like Qwen, ERNIE-Image, and many community favorites.

- **Datasets:** Explore a rich library of over 500,000 datasets, updated regularly and covering a wide range of machine learning tasks and domains.

- **Spaces:** Showcase and run interactive ML applications shared by the community. These include state-of-the-art voice cloning, image editing, video generation from text prompts, and more.

- **Compute & Enterprise Solutions:** Accelerate your machine learning projects with scalable paid compute resources and enterprise-grade solutions designed for team collaboration and production deployment.

---

## Our Community and Culture

Hugging Face thrives on its open, collaborative culture, promoting transparency and accessibility in AI development. It’s a home for innovators who believe in:

- **Open Collaboration:** Unlimited hosting and sharing of public models, datasets, and applications.
- **Building Open and Ethical AI:** Empowering users to contribute towards an AI ecosystem that is open, ethical, and beneficial for all.
- **Continuous Learning:** Supporting the next generation of ML engineers and scientists to build strong portfolios and grow their careers by sharing work globally.
- **Diverse Modalities:** Encouraging exploration across all data types and AI tasks, fostering creativity and innovation.

---

## Customers and Users

Hugging Face supports a broad spectrum of users, from individual machine learning practitioners and hobbyists to large enterprises and research institutions. The platform's tools and resources help accelerate AI development workflows across industries including tech, healthcare, finance, education, and more.

---

## Careers at Hugging Face

Join a dynamic team at the cutting edge of AI technology. Hugging Face offers exciting career opportunities for passionate individuals who want to help shape the future of machine learning. Whether your expertise lies in research, engineering, product development, or community engagement, you can contribute to a mission-driven company focused on open AI innovation.

Explore positions and be part of a diverse and inclusive workplace that values transparency, continuous learning, and collaboration.

---

## Get Involved

- Sign up for free to start exploring models, datasets, and applications.
- Share your own machine learning projects and build your portfolio.
- Participate in the fast-growing AI community that shapes the future of machine learning.

---

## Connect With Us

Visit us at [huggingface.co](https://huggingface.co)  
Explore ML Projects | Collaborate on Open Source | Empower AI Innovation

---

### Brand Highlights

- Recognizable logo and vibrant brand colors: Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280)
- Strong community focus and open-source commitment

---

**Hugging Face – The AI community building the future.**

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a vibrant and rapidly growing platform at the forefront of the machine learning (ML) community. It serves as a central hub where ML engineers, scientists, and enthusiasts collaborate to create, discover, and share open-source machine learning models, datasets, and applications.

With over **2 million models**, **500,000+ datasets**, and **1 million+ applications** available, Hugging Face empowers users to accelerate the development of AI technologies across diverse modalities — including text, images, video, audio, and even 3D.

---

## What Hugging Face Offers

- **Hugging Face Hub:**  
  A collaborative platform enabling anyone to host, share, explore, and experiment with ML models, datasets, and applications. Whether you are an individual developer or part of a large team, the Hub allows you to build and showcase your machine learning portfolio.

- **Open Source Stack:**  
  Tools and libraries that allow faster innovation, encouraging transparency, openness, and ethical AI development.

- **Spaces:**  
  A place for running and sharing AI applications; examples include voice cloning for 600+ languages, video generation from images, and running lightweight Large Language Models (LLMs) in browsers or on GPUs.

- **Enterprise & Compute Solutions:**  
  Paid services to support teams and businesses requiring cloud compute infrastructure and enterprise-grade solutions to scale AI projects efficiently.

---

## Company Culture & Community

- **Community-Driven:**  
  Hugging Face thrives on a collaborative spirit, building an open and ethical AI future together. The company’s platform is built around empowering users to learn, build, and share—fostering innovation through cooperation.

- **Open & Inclusive:**  
  Open source at its core, Hugging Face invites not just experts but also newcomers, making AI accessible to all levels of expertise.

- **Empowering Next Gen ML Engineers:**  
  The platform supports the growth of the machine learning community by enabling seamless sharing and exploration, helping individuals to build strong portfolios and reputations in AI.

---

## Customers & Users

Hugging Face serves a broad range of customers including:

- Individual machine learning engineers and researchers  
- Academic institutions and students  
- Startups and large enterprises working on AI innovation  
- Developers creating AI-powered products and services  
- Organizations seeking collaborative, open-source AI ecosystems

---

## Careers & Opportunities

Hugging Face is continually growing and offers exciting career opportunities for talented individuals passionate about AI and machine learning. Careers at Hugging Face focus on:

- Building open-source tools and infrastructure for ML innovation  
- Developing scalable AI solutions that serve the global community  
- Engaging in a culture centered on openness, ethics, and collaboration  

Interested candidates can explore roles in engineering, research, data science, product management, and more, joining a company that champions community and cutting-edge AI.

---

## Brand & Visual Identity

Hugging Face’s visual identity reflects its friendly, approachable, and innovative nature with bright colors such as:

- Primary Yellow: #FFD21E  
- Vibrant Orange: #FF9D00  
- Modern Gray: #6B7280  

The playful and engaging brand supports the mission of making AI accessible and fun.

---

## Join the Future of AI

Build, share, and explore with Hugging Face — the platform where the machine learning community creates the future of artificial intelligence.

Start your AI journey or accelerate your projects today:  
**[Sign Up](https://huggingface.co/join) | [Browse Models](https://huggingface.co/models) | [Explore AI Apps](https://huggingface.co/spaces)**

---

Hugging Face — Empowering the world to build an open, collaborative, and ethical AI future.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>

</td>
</tr>

</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>
